# AEGIS — Adaptive Security Gateway: ML Pipeline

**End-to-end:** load dataset → clean → EDA plots → train/test split → train 9 models →
metrics table (Accuracy / Precision / Recall / F1 / AUC) → confusion matrix → ROC → feature importance → 5-fold CV.

**Dataset:** `dataset/ml_dataset_fixed.csv` — 499,755 rows × 20 features (behavioural leakage removed)

| | |
|---|---|
| Label | `0` = benign (299,755) · `1` = attack (200,000) |
| attack_type | sqli, xss, xxe, ssrf, path_traversal, command_injection, ldap, nosql, prompt_injection, header_injection, open_redirect, csrf_bypass, decoy |
| Sources | CICIDS2017, NSL-KDD, UNSW-NB15, OWASP ModSecurity, HttpParamsDataset, PayloadsAllTheThings, JailbreakLLMs — see `research/papers_summary.md` |

### How to run (3 steps)
1. **Runtime → Run all** (or `Shift+Enter` cell by cell).
2. **Cell 2 opens your PC's file explorer** → select `dataset/ml_dataset_fixed.csv` (139 MB).
3. Everything after that runs automatically — plots, table, confusion matrix.

> Tip: for a quick demo set `SAMPLE = 100000` in the first code cell. `None` = full 499k run (~5-10 min).

In [ ]:
import os, glob, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TEST_SIZE    = 0.2
SAMPLE       = None          # None = full dataset, or e.g. 100000 for a quick demo
np.random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', 50)
print('pandas', pd.__version__, '| numpy', np.__version__)

In [ ]:
def find_csv():
    for p in ['/content/ml_dataset_fixed.csv',
              '/content/drive/MyDrive/ml_dataset_fixed.csv']:
        if os.path.exists(p):
            return p
    hits = glob.glob('/content/drive/MyDrive/**/ml_dataset_fixed.csv', recursive=True)
    return hits[0] if hits else None

CSV_PATH = find_csv()

if CSV_PATH is None:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        CSV_PATH = find_csv()
    except Exception as e:
        print('drive mount failed:', e)

if CSV_PATH is None:
    from google.colab import files
    uploaded = files.upload()
    CSV_PATH = list(uploaded.keys())[0]

print('dataset ->', CSV_PATH)

In [ ]:
df = pd.read_csv(CSV_PATH, low_memory=False)
print('raw shape :', df.shape)
print('columns   :', list(df.columns))
df.head()

## Step 1 — Cleaning
* coerce every feature + label to numeric (`errors='coerce'`)
* drop rows with missing values
* drop exact duplicate rows
* drop constant (zero-variance) columns
* keep only binary labels 0/1

In [ ]:
FEATURES = [
    'url_length', 'url_entropy', 'num_special_chars', 'num_encoded_chars',
    'has_sql_keywords', 'has_js_keywords', 'has_path_traversal', 'has_cmd_injection',
    'has_xxe_patterns', 'has_ssi_patterns', 'has_injection_patterns',
    'has_prompt_injection', 'has_suspicious_encoding',
    'body_length', 'body_entropy', 'num_headers', 'has_user_agent',
    'requests_per_minute', 'unique_urls_per_session', 'error_rate'
]
TARGET = 'label'

n0 = len(df)
for c in FEATURES + [TARGET]:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df = df.dropna(subset=FEATURES + [TARGET])
n1 = len(df)
df = df.drop_duplicates()
n2 = len(df)

dead = [c for c in FEATURES if df[c].nunique() <= 1]
FEATURES = [c for c in FEATURES if c not in dead]

df[TARGET] = df[TARGET].astype(int)
df = df[df[TARGET].isin([0, 1])]

print(f'rows            : {n0:,} -> {len(df):,}')
print(f'dropped NaN     : {n0 - n1:,}')
print(f'dropped dups    : {n1 - n2:,}')
print(f'constant cols   : {dead if dead else "none"}')
print(f'features kept   : {len(FEATURES)}')
print()
print('class balance:')
print(df[TARGET].value_counts().rename({0: 'benign', 1: 'attack'}))
print()
print('attack_type breakdown:')
print(df['attack_type'].value_counts())

## Step 2 — Exploratory Data Analysis (graphs)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(19, 4.5))

df[TARGET].map({0: 'benign', 1: 'attack'}).value_counts().plot.bar(
    ax=ax[0], color=['#4C72B0', '#C44E52'], rot=0)
ax[0].set_title('Class distribution')
ax[0].set_ylabel('requests')
for p in ax[0].patches:
    ax[0].text(p.get_x() + p.get_width() / 2, p.get_height() + 3000,
               f'{int(p.get_height()):,}', ha='center', fontsize=9)

vc = df['attack_type'].value_counts()
vc.plot.bar(ax=ax[1], color='#55A868', rot=60)
ax[1].set_title('Attack category breakdown')
ax[1].set_ylabel('requests')

corr = df[FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()
corr.plot.barh(ax=ax[2], color=np.where(corr > 0, '#C44E52', '#4C72B0'))
ax[2].set_title('Correlation with label')
ax[2].axvline(0, color='black', lw=0.8)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))

for lab, col in [(0, '#4C72B0'), (1, '#C44E52')]:
    sub = df[df[TARGET] == lab]
    ax[0].hist(sub['url_length'], bins=60, alpha=0.6, label=f'label={lab}', color=col)
ax[0].set_title('URL length distribution')
ax[0].set_xlabel('url_length')
ax[0].legend()

mean_by_class = df.groupby(TARGET)[FEATURES].mean().T
mean_by_class.plot.bar(ax=ax[1], color=['#4C72B0', '#C44E52'], rot=75)
ax[1].set_title('Mean feature value per class (scaled view)')
ax[1].legend(['benign', 'attack'])

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
m = df[FEATURES].corr()
im = ax.imshow(m, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(FEATURES)))
ax.set_yticks(range(len(FEATURES)))
ax.set_xticklabels(FEATURES, rotation=90, fontsize=8)
ax.set_yticklabels(FEATURES, fontsize=8)
for i in range(len(FEATURES)):
    for j in range(len(FEATURES)):
        ax.text(j, i, f'{m.iloc[i, j]:.2f}', ha='center', va='center', fontsize=6)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Feature correlation heatmap')
plt.tight_layout()
plt.show()

## Step 3 — Train / Test split + scaling
80/20, stratified, `random_state=42` (same split as `scripts/train_fixed.py`, so numbers match `results/fixed_report.txt`).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

work = df.sample(n=SAMPLE, random_state=RANDOM_STATE) if SAMPLE else df

X = work[FEATURES].values
y = work[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'samples      : {len(work):,}')
print(f'features     : {len(FEATURES)}')
print(f'train / test : {len(X_train):,} / {len(X_test):,}')
print(f'train labels : {np.bincount(y_train)}')
print(f'test  labels : {np.bincount(y_test)}')

## Step 4 — Train 9 models
Regularized / anti-overfit settings, identical to `scripts/train_fixed.py`.
Reported: train accuracy, test accuracy, **train-test gap** (overfit if > 0.03), precision, recall, F1, ROC-AUC.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score,
                             classification_report)
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                             ExtraTreesClassifier, AdaBoostClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=5,
                                            n_jobs=-1, random_state=RANDOM_STATE, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, min_samples_leaf=10,
                                            random_state=RANDOM_STATE, class_weight='balanced'),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=80, learning_rate=0.1,
                                                    max_depth=4, random_state=RANDOM_STATE),
    'Extra Trees': ExtraTreesClassifier(n_estimators=80, max_depth=12, min_samples_leaf=5,
                                        n_jobs=-1, random_state=RANDOM_STATE, class_weight='balanced'),
    'AdaBoost': AdaBoostClassifier(n_estimators=50, learning_rate=0.5, random_state=RANDOM_STATE),
    'Logistic Regression': LogisticRegression(max_iter=200, C=0.5, random_state=RANDOM_STATE,
                                              class_weight='balanced', n_jobs=-1),
    'Linear SVM': LinearSVC(max_iter=300, C=0.5, random_state=RANDOM_STATE, class_weight='balanced'),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=15, n_jobs=-1),
    'MLP Neural Network': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=80, alpha=0.01,
                                         random_state=RANDOM_STATE, early_stopping=True),
}
SCALED = {'Logistic Regression', 'Linear SVM', 'K-Nearest Neighbors', 'MLP Neural Network'}

fitted, results = {}, []
for name, model in models.items():
    Xtr, Xte = (X_train_s, X_test_s) if name in SCALED else (X_train, X_test)
    t0 = time.time()
    model.fit(Xtr, y_train)
    tt = time.time() - t0
    y_tr = model.predict(Xtr)
    y_te = model.predict(Xte)
    train_acc = accuracy_score(y_train, y_tr)
    acc  = accuracy_score(y_test, y_te)
    cm   = confusion_matrix(y_test, y_te)
    try:
        proba = model.predict_proba(Xte)[:, 1] if hasattr(model, 'predict_proba') \
                else model.decision_function(Xte)
        roc = roc_auc_score(y_test, proba)
    except Exception:
        roc = 0.0
    fitted[name] = model
    results.append({
        'model': name,
        'train_acc': train_acc, 'accuracy': acc,
        'gap': train_acc - acc,
        'precision': precision_score(y_test, y_te, zero_division=0),
        'recall': recall_score(y_test, y_te, zero_division=0),
        'f1': f1_score(y_test, y_te, zero_division=0),
        'roc_auc': roc, 'train_time_s': tt,
        'TP': int(cm[1, 1]), 'TN': int(cm[0, 0]),
        'FP': int(cm[0, 1]), 'FN': int(cm[1, 0]),
    })
    print(f'{name:22s} train={train_acc:.4f} test={acc:.4f} '
          f'gap={train_acc - acc:+.4f} F1={results[-1]["f1"]:.4f} '
          f'AUC={roc:.4f} time={tt:.1f}s')

print('\ntraining complete')

In [ ]:
table = (pd.DataFrame(results)
         .sort_values('f1', ascending=False)
         .reset_index(drop=True))
table.insert(0, 'rank', range(1, len(table) + 1))
table['overfit'] = np.where(table['gap'] > 0.03, 'OVERFIT', 'OK')

show = table.copy()
for c in ['train_acc', 'accuracy', 'gap', 'precision', 'recall', 'f1', 'roc_auc']:
    show[c] = show[c].map(lambda v: f'{v:.4f}')
show['train_time_s'] = show['train_time_s'].map(lambda v: f'{v:.1f}')

print('=' * 110)
print('AEGIS MODEL COMPARISON  —  Accuracy / Precision / Recall / F1 / AUC')
print('=' * 110)
display(show)

best = table.iloc[0]
print(f"\nBEST: {best['model']}  accuracy={best['accuracy']:.4f}  "
      f"precision={best['precision']:.4f}  recall={best['recall']:.4f}  "
      f"f1={best['f1']:.4f}  auc={best['roc_auc']:.4f}  gap={best['gap']:+.4f}")
print(f"Confusion: TP={best['TP']:,}  TN={best['TN']:,}  FP={best['FP']:,}  FN={best['FN']:,}")

table.to_csv('aegis_model_comparison.csv', index=False)
print('\nsaved -> aegis_model_comparison.csv')

In [ ]:
t = table.sort_values('accuracy')
x = np.arange(len(t))
w = 0.2

fig, ax = plt.subplots(figsize=(14, 5.5))
ax.bar(x - 1.5 * w, t['accuracy'], w, label='Accuracy', color='#4C72B0')
ax.bar(x - 0.5 * w, t['precision'], w, label='Precision', color='#55A868')
ax.bar(x + 0.5 * w, t['recall'], w, label='Recall', color='#C44E52')
ax.bar(x + 1.5 * w, t['f1'], w, label='F1', color='#8172B2')
ax.set_xticks(x)
ax.set_xticklabels(t['model'], rotation=35, ha='right')
ax.set_ylim(0, 1.12)
ax.set_ylabel('score')
ax.set_title('Model comparison — Accuracy / Precision / Recall / F1')
ax.legend(ncol=4, loc='upper left')
ax.grid(axis='y', alpha=0.3)
for i, v in enumerate(t['accuracy']):
    ax.text(i - 1.5 * w, v + 0.015, f'{v:.3f}', ha='center', fontsize=7)
plt.tight_layout()
plt.show()

## Step 5 — Confusion matrix + classification report (best model)

In [ ]:
best_name = table.iloc[0]['model']
best_model = fitted[best_name]
Xte = X_test_s if best_name in SCALED else X_test
y_pred = best_model.predict(Xte)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5.4, 4.4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted\nBenign', 'Predicted\nAttack'])
ax.set_yticklabels(['Actual\nBenign', 'Actual\nAttack'])
ax.set_title(f'Confusion Matrix — {best_name}')
for i in range(2):
    for j in range(2):
        v = cm[i, j]
        ax.text(j, i, f'{v:,}', ha='center', va='center', fontsize=17,
                color='white' if v > cm.max() / 2 else 'black')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=['benign', 'attack'], digits=4))

## Step 6 — ROC curves

In [ ]:
from sklearn.metrics import roc_curve, auc

fig, ax = plt.subplots(figsize=(8, 6.5))
for name, model in fitted.items():
    Xte = X_test_s if name in SCALED else X_test
    try:
        scores = model.predict_proba(Xte)[:, 1] if hasattr(model, 'predict_proba') \
                 else model.decision_function(Xte)
        fpr, tpr, _ = roc_curve(y_test, scores)
        ax.plot(fpr, tpr, lw=1.8, label=f'{name} (AUC={auc(fpr, tpr):.3f})')
    except Exception as e:
        print('skip', name, e)
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC curves — all models')
ax.legend(loc='lower right', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 7 — Feature importance (Random Forest)
Explains *why* the gateway scores a request — content features must dominate now that behavioural leakage was removed.

In [ ]:
rf = fitted['Random Forest']
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
imp.plot.barh(ax=ax, color=np.where(imp.values > imp.mean(), '#C44E52', '#4C72B0'))
ax.set_title('Random Forest feature importance')
ax.set_xlabel('importance')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(imp.sort_values(ascending=False).head(12).to_string())

## Step 8 — 5-fold cross-validation (generalization proof)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv_models = {
    'Random Forest': RandomForestClassifier(n_estimators=50, max_depth=12, min_samples_leaf=5,
                                            n_jobs=-1, random_state=RANDOM_STATE, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, min_samples_leaf=10,
                                            random_state=RANDOM_STATE, class_weight='balanced'),
    'Logistic Regression': LogisticRegression(max_iter=200, C=0.5, random_state=RANDOM_STATE,
                                              class_weight='balanced', n_jobs=-1),
}
n_cv = min(30000, len(X))
if n_cv >= len(X):
    Xs, ys = X, y
else:
    Xs, _, ys, _ = train_test_split(X, y, train_size=n_cv, random_state=RANDOM_STATE, stratify=y)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_rows = []
for name, m in cv_models.items():
    sc = cross_val_score(m, Xs, ys, cv=skf, scoring='f1', n_jobs=-1)
    cv_rows.append({'model': name, 'f1_mean': sc.mean(), 'f1_std': sc.std(),
                    'folds': [round(float(v), 4) for v in sc]})
    print(f'{name:22s} folds={[round(float(v), 4) for v in sc]} '
          f'mean={sc.mean():.4f} std={sc.std():.4f}')

cv_table = pd.DataFrame(cv_rows)
cv_table.to_csv('aegis_cv_results.csv', index=False)
cv_table

---
## Done — how this maps back to the repo

| This notebook | Repo file |
|---|---|
| cleaning | `scripts/fix_overfitting.py` → `dataset/ml_dataset_fixed.csv` |
| training + metrics | `scripts/train_fixed.py` |
| saved tables | `results/fixed_report.txt`, `results/fixed_metrics.csv`, `results/fixed_results.json` |
| final exported model | `scripts/export_model.py` → `models/aegis_model.pkl` |
| sources / papers | `research/papers_summary.md` |

**Talking points for the mentor**
1. First run gave *perfect* 1.0000 accuracy → diagnosed as **data leakage** (`risk_score`/`severity` derived from the label, plus behavioural features that didn't overlap between classes).
2. Fixed by removing label-derived columns and resampling behavioural features from shared distributions → honest **~0.885 accuracy, F1 ~0.83, train-test gap -0.0004** (no overfitting).
3. 5-fold CV std ~0.002 → generalization is stable.
4. Gateway uses RandomForest (88.5%) over MLP (90.8%) for **5x faster inference + explainability** (feature contribution + counterfactual in the dashboard).